# Time Series Forecasting

In [ ]:
import pandas as pd

eskom = pd.read_csv("eskom_clean.csv", parse_dates = ["date_time"])

eskom = eskom.set_index("date_time").sort_index()

print(f"Index type: {type(eskom.index)}")
print(f"Index start: {eskom.index.min()}")
print(f"Index end: {eskom.index.max()}")
print(f"Frequency: {eskom.index.freq}")
print(f"Shape: {eskom.shape}")

In [ ]:
# set explicity hourly frequency
eskom = eskom.asfreq("h")

# check for any gaps that appeared after setting frequency
missing_after = eskom["residual_demand"].isna().sum()
print(f"Frequency: {eskom.index.freq}")
print(f"Missing values after setting frequency: {missing_after}")
print(f"Shape: {eskom.shape}")

## Visualise any patterns

In [ ]:
import matplotlib.dates as mdates
import matplotlib.pyplot as plt
from statsmodels.tsa.seasonal import seasonal_decompose 

fig, axes = plt.subplots(4, 1, figsize = (14, 14))

# full demand series plot
daily_demand = eskom["residual_demand"].resample("D").mean()
axes[0].plot(daily_demand.index, daily_demand.values, color = "#D85A30", lw = 0.8, alpha = 0.8)
axes[0].set_title("Daily Average Demand: Overall Trend (2021-2026)", fontsize = 12, fontweight = "bold")
axes[0].set_ylabel("MW")
axes[0].xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
axes[0].xaxis.set_major_locator(mdates.YearLocator())

# annual seasonality plot
monthly_avg = eskom.groupby(eskom.index.month)["residual_demand"].mean()
month_names = ["Jan", "Feb", "Mar", "Apr", "May", "Jun", "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"]
axes[1].bar(month_names, monthly_avg.values, color = "#378ADD", edgecolor = "none", alpha = 0.85)
axes[1].set_title("Average Demand by Month: Annual Seasonality", fontsize = 12, fontweight = "bold")
axes[1].set_ylabel("MW")

# diurnal pattern plot 
hourly_avg = eskom.groupby(eskom.index.hour)["residual_demand"].mean()
axes[2].plot(hourly_avg.index, hourly_avg.values, color = "#1D9E75", lw = 2, marker = "o", markersize = 4)
axes[2].set_title("Average Demand by Hour of Day: Diurnal Pattern", fontsize = 12, fontweight = "bold")
axes[2].set_ylabel("MW")
axes[2].set_xlabel("Hour of Day")
axes[2].set_xticks(range(0, 24, 1))

# use weekly resampled data to make decomposition manageable
weekly = eskom["residual_demand"].resample("W").mean()
decomp = seasonal_decompose(weekly, model = "additive", period = 52)
axes[3].plot(decomp.resid.index, decomp.resid.values, color = "#7F77DD", lw = 0.8, alpha = 0.8)
axes[3].axhline(y = 0, color = "black", linestyle = "--", lw =1)
axes[3].set_title("Residuals: Random Noise After Removing Trend and Seasonality", fontsize = 12, fontweight = "bold")
axes[3].set_ylabel("MW")
axes[3].xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
axes[3].xaxis.set_major_locator(mdates.YearLocator())

plt.tight_layout()
plt.savefig("vis8_time_series_decomposition.png", bbox_inches = "tight")
plt.show()

## Augmented Dickey-Fuller Test

In [ ]:
from statsmodels.tsa.stattools import adfuller

daily_demand = eskom["residual_demand"].resample("D").mean()
result = adfuller(daily_demand.dropna())

print(f"ADF Statistic: {result[0]:.4f}")
print(f"p-value: {result[1]:.4f}")
print(f"Critical Values: ")
for key, value in result[4].items():
    print(f"{key}: {value:.4f}")
    
if result[1] < 0.05:
    print("\nConclusion: Series IS stationary (p < 0.05)")
else:
    print("\nConclusion: Series is NOT stationary (p > 0.05)")

## Differencing 

In [ ]:
daily_demand_diff = daily_demand.diff(365).dropna()

result_diff = adfuller(daily_demand_diff.dropna())

print(f"ADF Statistic: {result_diff[0]:.4f}")
print(f"p-value: {result_diff[1]:.4f}")

if result_diff[1] < 0.05:
    print("\nConclusion: Series IS stationary (p < 0.05)")
else:
    print("\nConclusion: Series is NOT stationary (p > 0.05)")

In [ ]:
daily_demand_diff2 = daily_demand.diff(365).diff(1).dropna()
fig, axes = plt.subplots(3, 1, figsize = (14, 12))

axes[0].plot(daily_demand.index, daily_demand.values, color = "#D85A30", lw = 0.8)
axes[0].set_title("Original Daily Demand", fontsize = 12, fontweight = "bold")
axes[0].set_ylabel("MW")

axes[1].plot(daily_demand_diff.index, daily_demand_diff.values, color = "#1D9E75", lw = 0.8)
axes[1].axhline(y = 0, color = "black", linestyle = "--", lw = 1)
axes[1].set_title("After Seasonal Differencing Only", fontsize = 12, fontweight = "bold")
axes[1].set_ylabel("Change in MW")

axes[2].plot(daily_demand_diff2.index, daily_demand_diff2.values, color = "#378ADD", lw = 0.8)
axes[2].axhline(y = 0, color = "black", linestyle = "--", lw =1)
axes[2].set_title("After Seasonal + Regular Differencing", fontsize = 12, fontweight = "bold")
axes[2].set_ylabel("Change in MW")

plt.tight_layout()
plt.savefig("vis9_differenced_demand.png", bbox_inches = "tight")
plt.show()

In [ ]:
import numpy as np

print("Basic statistics of daily demand:")
print(daily_demand.describe())

print(f"\nAny NaN values: {daily_demand.isna().sum()}")
print(f"Any infinite values: {np.isinf(daily_demand).sum()}")
print(f"Any zero values: {(daily_demand == 0).sum()}")

for year in daily_demand.index.year.unique():
    year_data = daily_demand[daily_demand.index.year == year]
    print(f"{year} - mean: {year_data.mean():.1f} MW, std: {year_data.std():.1f} MW")

In [ ]:
daily_demand_clean = daily_demand.copy()
daily_demand_clean = daily_demand_clean.replace(0, np.nan)
daily_demand_clean = daily_demand_clean.fillna(method = "ffill")

daily_demand_clean = daily_demand_clean[daily_demand_clean.index.year < 2026]

print(f"Rows after cleaning: {len(daily_demand_clean)}")
print(f"Zero values remaning: {(daily_demand_clean == 0).sum()}")
print(f"Years remaining: {daily_demand_clean.index.year.unique().tolist()}")
print(f"\nAnnual means after cleaning:")

for year in daily_demand_clean.index.year.unique():
    year_data = daily_demand_clean[daily_demand_clean.index.year == year]
    print(f"{year} - mean: {year_data.mean():.1f} MW, std: {year_data.std():.1f} MW") 
    
result_clean = adfuller(daily_demand_clean.dropna())
print(f"\nADF Statistic: {result_clean[0]:.4f}")
print(f"p-value: {result_clean[1]:.4f}")

if result_clean[1] < 0.05:
    print("\nConclusion: Series IS stationary (p < 0.05)")
else:
    print("\nConclusion: Series is NOT stationary (p > 0.05)")

In [ ]:
from prophet import Prophet 

prophet_df = daily_demand_clean.reset_index()
prophet_df.columns = ["ds", "y"]

print(f"Shape: {prophet_df.shape}")
print(f"Columns: {prophet_df.columns.tolist()}")
print(f"Date range: {prophet_df["ds"].min()} -> {prophet_df["ds"].max()}")
print(prophet_df.head())

## Train/Test Split

In [ ]:
train = prophet_df[prophet_df["ds"] < "2025-01-01"]
test = prophet_df[prophet_df["ds"] >= "2025-01-01"]

print(f"Training set: {train.shape[0]} days ({train['ds'].min().date()} -> {train['ds'].max().date()}) ")
print(f"Test set: {test.shape[0]} days ({test['ds'].min().date()} -> {test['ds'].max().date()})")

## Train Prophet Model

In [ ]:
model = Prophet(
    yearly_seasonality = True,
    weekly_seasonality = True,
    daily_seasonality = False,
    changepoint_prior_scale = 0.05, # control how flexible the trend is (lower = smooth trend, higher = more flexible)
    seasonality_mode = "additive"
)

# add SA public holidays
model.add_country_holidays(country_name = "ZA")

# train on training set
model.fit(train)
print("Model successfully trained")

## Make Predictions

In [ ]:
future = model.make_future_dataframe(periods = 365, freq = "D")
forecast = model.predict(future)

# only extract test period predictions
forecast_test = forecast[forecast["ds"] >= "2025-01-01"][["ds", "yhat", "yhat_lower", "yhat_upper"]]
forecast_test = forecast_test.reset_index(drop = True)

print(f"Forecast shape: {forecast_test.shape}")
print("Columns:")
print("ds - the date")
print("yhat - the predicted demand value")
print("yhat_lower - lower bound of uncertainty interval")
print("yhat_upper - upper bound of uncertainty interval")
print("\nFirst 5 predictions:")
print(forecast_test.head())


## Model Evaluation 

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error

# merge actual and predicted
evaluation = test.merge(forecast_test, on = "ds", how = "left")

# calculate error metrics 
mae = mean_absolute_error(evaluation["y"], evaluation["yhat"])
rmse = np.sqrt(mean_squared_error(evaluation["y"], evaluation["yhat"]))
mape = (np.abs((evaluation["y"] - evaluation["yhat"]) / evaluation["y"])).mean() * 100

print("Model Evaluation on Test Set (2025): ")
print(f"MAE: {mae:.1f} MW")
print(f"RMSE: {rmse:.1f} MW")
print(f"MAPE: {mape:.2f}%")
print(f"\nMean actual demand: {evaluation['y'].mean():.1f} MW")
print(f"Mean predicted: {evaluation['yhat'].mean():.1f} MW")

## Visualise actual vs predicted

In [ ]:
fig, axes = plt.subplots(2, 1, figsize = (14, 10))

# actual vs predicted with confidence interval plot 
axes[0].plot(evaluation["ds"], evaluation["y"], color = "#D85A30", lw = 1.5, label = "Actual Demand")
axes[0].plot(evaluation["ds"], evaluation["yhat"], color = "#378ADD", lw = 1.5, linestyle = "--", label = "Predicted Demand")
axes[0].fill_between(evaluation["ds"], evaluation["yhat_lower"], evaluation["yhat_upper"], alpha = 0.2, color ="#378ADD", label = "80% Confidence Interval")
axes[0].set_title("Actual vs Predicted Daily Demand: Test Set 2025", fontsize = 13, fontweight = "bold", pad = 15)
axes[0].set_ylabel("Demand (MW)")
axes[0].legend(fontsize = 9)
axes[0].xaxis.set_major_formatter(mdates.DateFormatter("%b %Y"))
axes[0].xaxis.set_major_locator(mdates.MonthLocator(interval = 2))

# prediction error over time
evaluation["error"] = evaluation["y"] - evaluation["yhat"]
axes[1].bar(evaluation["ds"], evaluation["error"], color = evaluation["error"].apply(lambda x: "#1D9E75" if x >= 0 else "#D85A30"), alpha = 0.7, width = 1)
axes[1].axhline(y = evaluation["error"].mean(), color = "#7F77DD", linestyle = "--", lw = 1.5, label = f"Mean error: {evaluation['error'].mean():.0f} MW")
axes[1].set_title("Prediction Error Over Time (Actual - Predicted)", fontsize = 13, fontweight = "bold", pad = 15)
axes[1].set_ylabel("Error (MW)")
axes[1].legend(fontsize = 9)
axes[1].xaxis.set_major_formatter(mdates.DateFormatter("%b %Y"))
axes[1].xaxis.set_major_locator(mdates.MonthLocator(interval = 2))

plt.tight_layout()
plt.savefig("vis10_actual_vs_predicted.png", bbox_inches = "tight")
plt.show()

## Forecasting Future Demand

In [ ]:
future_extended = model.make_future_dataframe(periods = 365 * 6, freq = "D")
forecast_extended = model.predict(future_extended)

# only extract the future period 
future_only = forecast_extended[forecast_extended["ds"] > "2025-12-31"][["ds", "yhat", "yhat_lower", "yhat_upper"]]

# annual avg forecast 
future_only["year"] = pd.to_datetime(future_only["ds"]).dt.year
annual_forecast = future_only.groupby("year").agg(
    avg_demand_mw = ("yhat", "mean"),
    avg_lower_mw = ("yhat_lower", "mean"),
    avg_upper_mw = ("yhat_upper", "mean"),
    peak_demand_mw = ("yhat", "max")
).reset_index()

print("Forecasted Annual Demand:")
print(annual_forecast.round(1).to_string(index = False))

fig, ax = plt.subplots(figsize = (14, 6))

# actual historical
ax.plot(prophet_df["ds"], prophet_df["y"], color = "#D85A30", lw = 1.2, label = "Historical Demand", alpha = 0.8)

# forecast 
ax.plot(forecast_extended["ds"], forecast_extended["yhat"], color = "#378ADD", lw = 1.5, linestyle = "--", label = "Forecasted Demand")
ax.fill_between(forecast_extended["ds"], forecast_extended["yhat_lower"], forecast_extended["yhat_upper"], alpha = 0.15, color = "#378ADD", label = "80% Confidence Interval")

# mark train/test boundry
ax.axvline(x = pd.Timestamp("2025-01-01"), color = "gray", linestyle = ":", lw = 1.5, label = "Train/Test Split")

# mark where forecasted begins 
ax.axvline(x = pd.Timestamp("2026-01-01"), color = "#1D9E75", linestyle = ":", lw = 1.5, label = "Forecast Start")

ax.set_title("Electricity Demand Forecast 2026-2028\n(Prophet Model)", fontsize = 13, fontweight = "bold", pad = 15)
ax.set_ylabel("Daily Average Demand (MW)")
ax.legend(fontsize = 9)
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
ax.xaxis.set_major_locator(mdates.YearLocator())

plt.tight_layout()
plt.savefig("vis11_demand_forecast.png", bbox_inches = "tight")
plt.show()  

## Apply Correction Factor (approximately 1773 MW on avg)

In [ ]:
# calculate the bias correction from test set evaluation
bias_correction = evaluation["error"].mean()
print(f"Bias correction factor: {bias_correction:.1f} MW")

# apply correction to forecast
future_all = forecast_extended[forecast_extended["ds"].dt.year.isin([2026, 2027, 2028, 2029, 2030])].copy()
future_all["yhat_corrected"] = future_all["yhat"] + bias_correction
future_all["yhat_lower_corrected"] = future_all["yhat_lower"] + bias_correction
future_all["yhat_upper_corrected"] = future_all["yhat_upper"] + bias_correction
future_all["year"] = future_all["ds"].dt.year

# recalculate annual forecast with correction
annual_all = future_all.groupby("year").agg(
    avg_demand_mw = ("yhat_corrected", "mean"),
    avg_lower_mw = ("yhat_lower_corrected", "mean"),
    avg_upper_mw = ("yhat_upper_corrected", "mean"),
    peak_demand_mw = ("yhat_corrected", "max")
).reset_index()

print("\nForecast comparison (with bias correction):")
print(annual_all.round(1).to_string(index = False))

## Save Correction Factor Forecast Figures

In [ ]:
# save annual summary separately
# annual_all.to_csv("annual_demand_forecast.csv", index = False)
# print("Saved: annual_demand_forecast.csv")
annual_all.to_excel("annual_demand_forecast.xlsx", index = False)
print("Saved: annual_demand_forecast.xlsx")